# Oblsk Negotiation Agent — walkthrough

An agent that prices every creator deal and negotiates like a real person, with a human approving each reply until autonomy is earned.

Two parts meet at one handoff:

- **the calculator** (`ev_engine` + `pricing`) fits a heavy-tailed model of the creator's views, runs a Monte Carlo, and builds the pricing ladder: **anchor** (where we open), **target** (the fee that hits our ROI goal), **walk-away** (the ceiling the downside still justifies);
- **the negotiator** (`behavior_tree` + `prose` + `qa`) runs the conversation: one pass per message, first branch that fits wins, every dollar read off the ladder.

An LLM writes the words and reads incoming messages when `ANTHROPIC_API_KEY` is set; it never invents a price or picks the move. Everything below runs offline.

In [ ]:
# In Colab: clone the repo and install deps first
# !git clone https://github.com/Sophie-S-Z/oblsk-negotiator.git
# %cd oblsk-negotiator
# !pip -q install numpy scipy pyyaml matplotlib
import os
os.environ['OBLSK_NO_LLM'] = '1'   # deterministic templates; unset to use the LLM
import numpy as np

## 1. The pricing thesis

A creator's next video will not get their average views: most posts land near the median, a few run far past it. The calculator fits a log-normal body with a Pareto tail and simulates thousands of outcomes, then derives the ladder from the distribution — the downside (p10) sets the walk-away, the winsorized expectation sets the target.

In [ ]:
from oblsk_negotiator import CreatorEconomics, fit_view_model, price_ladder, PricingPolicy

rng = np.random.default_rng(7)
views = np.concatenate([rng.lognormal(np.log(55000), 0.6, 24), [400000, 900000]])
vm = fit_view_model(views)
econ = CreatorEconomics(conversion_rate=0.0016, ltv_usd=80)

ladder = price_ladder(vm, econ, PricingPolicy())
print(vm.notes)
print(ladder.summary())

In [ ]:
import matplotlib.pyplot as plt

samples = vm.sample(20000, np.random.default_rng(1)) * econ.revenue_per_view
plt.figure(figsize=(9, 4))
plt.hist(np.clip(samples, 0, np.quantile(samples, 0.99)), bins=80, alpha=0.7)
for x, label in [(ladder.anchor, 'anchor'), (ladder.target, 'target'),
                 (ladder.walk_away, 'walk-away')]:
    plt.axvline(x, ls='--', lw=1.5, label=f'{label} ${x:,.0f}')
plt.legend(); plt.xlabel('revenue per video ($)'); plt.title('One video, simulated — and the ladder it implies')
plt.show()

## 2. One negotiation, human in the loop

A rule-based simulated creator plays the other side. Every outbound message passes through the approval gate; the transcript shows the move, the words, and the rationale.

In [ ]:
from oblsk_negotiator import CampaignContext, CampaignBrief, SimCreator, run_negotiation

creator = SimCreator(reservation_per_video=2450, opens_with='question',
                     questions=['What would the deliverables be?', 'And the timeline?'],
                     counter_ratio=0.6, bulk_tolerance=0.06, rng_seed=3)
outcome = run_negotiation(vm, econ, CampaignContext(), creator,
                          brief=CampaignBrief(brand='Aurora Skincare', product='the daily SPF serum'),
                          creator_name='Maya', verbose=True)
print()
print('status:', outcome.status, '| final:', outcome.final_total, '| rounds:', outcome.rounds)

## 3. A hard creator: revise once, restructure, escalate

When the floor is above our target the agent revises once (to the target, never the ceiling), reshapes into a bundle, and hands stalled threads to a person.

In [ ]:
hard = SimCreator(reservation_per_video=4200, counter_ratio=0.2, bulk_tolerance=0.0, rng_seed=0)
outcome = run_negotiation(vm, econ, CampaignContext(), hard, creator_name='Jordan', verbose=True)
print()
print('status:', outcome.status)

## 4. Replay a real thread

The practical test: shadow-run the agent over an email thread the team handled manually — a Gmail paste, signatures and all. At each creator message the report shows how the agent read it, the move it would have made, and what the team actually sent. Nothing is sent anywhere.

Watch for the human-in-the-loop moves in the report:

- **hold_firm** — a follow-up with no counter gets the standing offer restated, never an unforced concession;
- **propose_call** — when they want a call, the agent proposes the campaign's call windows and flags a teammate to send the invite and run it (`NEEDS A HUMAN`), with instructions to note what was agreed back into the thread;
- **ask_human** — a question the brief can't answer, or a reference to a call the agent wasn't on, pauses the thread and asks the team for exactly what's missing instead of bluffing;
- **escalate_human** — contract-term negotiation (exclusivity, equity, kill fees) hands the whole thread to a person.

The campaign file carries the brief (from the UNest content guide + program agreement), the economics, the negotiation stance, call windows, and how to recognize our side of the thread.

In [ ]:
from oblsk_negotiator.campaign import load_campaign
from oblsk_negotiator import replay_thread

camp = load_campaign('examples/unest_campaign.yaml')
thread = open('examples/unest_thread.txt', encoding='utf-8').read()

# Price the same creator the team was pricing (~150k median views/post here).
vm_creator = fit_view_model(np.random.default_rng(2).lognormal(np.log(150000), 0.6, 24))
report = replay_thread(thread, vm_creator, camp.econ, camp.ctx,
                       brief=camp.brief, us_aliases=camp.us_aliases, creator_name='Karissa')
print(report.report())

When the agent is missing context only the team has, it asks instead of guessing:

In [ ]:
from oblsk_negotiator import NegotiationState, decide
from oblsk_negotiator.replay import heuristic_interpret

state = NegotiationState('c', 'unest', 't1')
msg = heuristic_interpret('Is UNest FDIC insured? My followers will ask.')
d = decide(msg, state, vm_creator, camp.econ, camp.ctx, brief=camp.brief)
print('action:', d.action.value)
print('asks the team:', d.human_prompt)

## 5. Metrics across many creators

In [ ]:
from oblsk_negotiator import batch_metrics

outcomes = []
for seed in range(20):
    r = np.random.default_rng(seed)
    c = SimCreator(reservation_per_video=float(r.uniform(1700, 3100)),
                   counter_ratio=float(r.uniform(0.3, 0.75)),
                   bulk_tolerance=float(r.uniform(0, 0.12)), rng_seed=seed)
    v = fit_view_model(r.lognormal(np.log(r.uniform(35000, 75000)), 0.62, 24))
    outcomes.append(run_negotiation(v, econ, CampaignContext(), c))
print(batch_metrics(outcomes).report())

## Where to go next

- `examples/unest_campaign.yaml` — copy per campaign; every knob the negotiator uses lives there.
- Set `ANTHROPIC_API_KEY` (and drop the `OBLSK_NO_LLM` line above) to have Claude draft the messages and read real threads.
- `py demo.py --campaign ... --replay your_thread.txt` runs section 4 from the terminal.
- Calibrate `economics` (conversion, LTV) and the CPM table in `pricing.py` against closed deals before trusting absolute dollars.